<a href="https://colab.research.google.com/github/riccardogs/PyTorch_tutorial/blob/main/Build_the_Neural_Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Neural networks comprise of layers/modules that perform operations on data. The torch.nn namespace provides all the building blocks you need to build your own neural network. Every module in PyTorch subclasses the nn.Module. A neural network is a module itself that consists of other modules (layers). This nested structure allows for building and managing complex architectures easily.

In the following sections, we’ll build a neural network to classify images in the FashionMNIST dataset.

In [15]:
import os
# import os: importa il modulo operativo di sistema
# - Serve per interagire con il file system
# - Utile per salvare modelli, caricare dati, gestire percorsi

import torch

from torch import nn
# from torch import nn: importa il modulo delle reti neurali
# - nn sta per "neural networks"
# - Contiene:
#   * Layer predefiniti: nn.Linear, nn.Conv2d, nn.LSTM, ecc.
#   * Funzioni di attivazione: nn.ReLU, nn.Sigmoid, nn.Softmax
#   * Funzioni di loss: nn.MSELoss, nn.CrossEntropyLoss
#   * Container: nn.Sequential, nn.Module (classe base per modelli)

from torch.utils.data import DataLoader
# from torch.utils.data import DataLoader: importa il caricatore di dati
# - DataLoader gestisce il caricamento in batch
# - Shuffle dei dati
# - Caricamento parallelo con più worker
# - Iterazione efficiente sui dataset

from torchvision import datasets, transforms
# from torchvision import datasets, transforms: importa moduli di torchvision
# - torchvision: pacchetto per computer vision
#
# - datasets: contiene dataset predefiniti
#   * FashionMNIST, MNIST, CIFAR10, CIFAR100, ImageNet, ecc.
#   * Scarica automaticamente i dati se non presenti
#   * Già suddivisi in training/test
#
# - transforms: contiene trasformazioni per immagini
#   * ToTensor(): converte immagini in tensor e normalizza
#   * Normalize(): normalizza con media e std dev
#   * Resize(): ridimensiona immagini
#   * RandomCrop(), RandomHorizontalFlip(): data augmentation
#   * Compose(): combina più trasformazioni

# Get Device for Training

We want to be able to train our model on an accelerator such as CUDA, MPS, MTIA, or XPU. If the current accelerator is available, we will use it. Otherwise, we use the CPU.

In [16]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
# device = ... : determina il dispositivo da usare (GPU o CPU)

# ANALISI PASSO PER PASSO:

# 1. torch.accelerator.is_available()
#    - Controlla se c'è un acceleratore disponibile (GPU NVIDIA, GPU Apple, ecc.)
#    - Restituisce True se presente, False se solo CPU

# 2. Se is_available() è True:
#    torch.accelerator.current_accelerator().type
#    - torch.accelerator.current_accelerator(): ottiene l'acceleratore corrente
#    - .type: restituisce il tipo di acceleratore come stringa
#      * "cuda" per GPU NVIDIA
#      * "mps" per GPU Apple (Metal Performance Shaders)
#      * Altri tipi possibili: "xpu", "mtia", ecc.

# 3. Se is_available() è False:
#    "cpu"  # usa la CPU come fallback

# 4. L'operatore ternario if/else in una riga:
#    valore_se_vero if condizione else valore_se_falso

print(f"Using {device} device")
# print(f"Using {device} device"): stampa il dispositivo che verrà utilizzato
# - Output esempio: "Using cuda device" (se GPU NVIDIA disponibile)
# - Output esempio: "Using mps device" (se Mac con chip M1/M2)
# - Output esempio: "Using cpu device" (se nessuna GPU disponibile)


Using cpu device


# Define the Class

We define our neural network by subclassing nn.Module, and initialize the neural network layers in __ init__. Every nn.Module subclass implements the operations on input data in the forward method.

In [17]:
class NeuralNetwork(nn.Module):
    # class NeuralNetwork(nn.Module): definisce una nuova classe che eredita da nn.Module
    # - nn.Module è la classe base per tutti i modelli di reti neurali in PyTorch
    # - Ereditare da nn.Module ci dà accesso a funzionalità come:
    #   * Gestione automatica dei parametri
    #   * Metodi come .to(device), .train(), .eval()
    #   * Salvataggio e caricamento con .state_dict()


    def __init__(self):
        # __init__: costruttore della classe, definisce l'architettura del modello
        # Qui dichiariamo TUTTI i layer che useremo

        super().__init__()
        # super().__init__(): chiama il costruttore della classe padre (nn.Module)
        # Necessario per inizializzare correttamente il modulo PyTorch

        self.flatten = nn.Flatten()
        # self.flatten = nn.Flatten(): layer che appiattisce le immagini
        # - Input: tensor di forma (batch_size, 1, 28, 28)
        # - Output: tensor di forma (batch_size, 28*28 = 784)
        # - Trasforma l'immagine 2D in un vettore 1D per i layer lineari

        self.linear_relu_stack = nn.Sequential(
            # nn.Sequential(): contenitore che applica i layer in sequenza
            # L'output di un layer diventa input del successivo

            nn.Linear(28*28, 512),
            # Primo layer lineare (fully connected)
            # - 28*28 = 784: dimensione dell'input (pixel dell'immagine appiattita)
            # - 512: dimensione dell'output (numero di neuroni)
            # - Ogni neurone è connesso a tutti i 784 pixel di input

            nn.ReLU(),
            # Funzione di attivazione ReLU (Rectified Linear Unit)
            # - Formula: f(x) = max(0, x)
            # - Introduce non-linearità nel modello
            # - Aiuta la rete a imparare relazioni complesse

               # ESEMPIO CONCRETO:
               # Immaginiamo che il layer lineare produca questi 5 valori (in realtà sono 512):
               # output_lineare = [-2.5, 1.3, -0.7, 4.2, -3.1]

               # Dopo ReLU:
               # output_relu = [0.0, 1.3, 0.0, 4.2, 0.0]
               # - -2.5 → 0.0 (negativo diventa zero)
               # - 1.3 → 1.3 (positivo rimane)
               # - -0.7 → 0.0 (negativo diventa zero)
               # - 4.2 → 4.2 (positivo rimane)
               # - -3.1 → 0.0 (negativo diventa zero)

               # VISUALIZZAZIONE GRAFICA:
               #    ↑
               # 4  |        ●
               # 3  |
               # 2  |
               # 1  |     ●
               # 0  |  ●     ●     ●
               #    +----------------→
               #     -2  -1   0   1   2   3   4
               #
               # La ReLU "spegne" i neuroni con output negativo
               # Questo introduce non-linearità e aiuta la rete a imparare pattern complessi

            nn.Linear(512, 512),
            # Secondo layer lineare
            # - Input: 512 (dal layer precedente)
            # - Output: 512 (ancora 512 neuroni)
            # - Layer nascosto (hidden layer)

            # PERCHÉ SI CHIAMANO "NASCOSTI" (HIDDEN):
            # - Non sono direttamente accessibili dall'esterno
            # - Non vediamo direttamente cosa rappresentano
            # - L'utente vede solo input (immagini) e output (classi)
            # - I layer nascosti sono "scatole nere" che trasformano i dati internamente

            nn.ReLU(),
            # Ancora una ReLU dopo il secondo layer

            nn.Linear(512, 10),
            # Terzo layer lineare (layer di output)
            # - Input: 512
            # - Output: 10 (numero di classi in Fashion-MNIST)
            # - Produce i "logits" (punteggi grezzi per ogni classe)
        )

    def forward(self, x):
        # forward: definisce il passaggio in avanti (forward pass)
        # - È il cuore della rete neurale: descrive come i dati fluiscono attraverso il modello
        # - Questo metodo VIENE CHIAMATO AUTOMATICAMENTE quando fai model(x)
        # - NON devi chiamarlo direttamente come model.forward(x) (anche se tecnicamente funziona)

        # Parametro x:
        # - x è il tensor di input (tipicamente un batch di immagini)
        # - Shape di x: (batch_size, canali, altezza, larghezza)
        # - Per Fashion-MNIST: (64, 1, 28, 28) con batch_size=64
        # - 64 immagini in scala di grigi (1 canale) di 28x28 pixel

        x = self.flatten(x)
        # x = self.flatten(x): appiattisce le immagini 2D in vettori 1D
        # - self.flatten è un layer nn.Flatten() definito in __init__
        # - COSA FA: prende ogni immagine e la "stende" in una lunga riga
        # - Prima: (64, 1, 28, 28) → 64 immagini, ognuna è una matrice 28x28
        # - Dopo: (64, 784) → 64 immagini, ognuna è un vettore di 784 numeri
        # - 784 = 28 * 28 (tutti i pixel in fila)
        # - Perché? I layer lineari (Linear) lavorano con vettori, non con matrici 2D

        logits = self.linear_relu_stack(x)
        # logits = self.linear_relu_stack(x): passa attraverso i layer sequenziali
        # - self.linear_relu_stack è il contenitore nn.Sequential definito in __init__
        # - Contiene: Linear(784,512) → ReLU → Linear(512,512) → ReLU → Linear(512,10)

        # COSA SUCCEDE QUI (passo dopo passo):
        # 1. Primo Linear(784,512):
        #    - Input: (64, 784)
        #    - Output: (64, 512) - 512 valori per ogni immagine
        #    - Ogni valore è una combinazione lineare dei 784 pixel

        # 2. ReLU:
        #    - Input: (64, 512)
        #    - Output: (64, 512) - stessa forma, ma valori negativi azzerati
        #    - max(0, x) applicato a ogni singolo valore

        # 3. Secondo Linear(512,512):
        #    - Input: (64, 512) (dopo ReLU)
        #    - Output: (64, 512) - trasformazione lineare

        # 4. ReLU:
        #    - Input: (64, 512)
        #    - Output: (64, 512) - ancora attivazione

        # 5. Terzo Linear(512,10):
        #    - Input: (64, 512)
        #    - Output: (64, 10) - 10 valori per ogni immagine (uno per classe)

        # Il risultato finale sono i "logits":
        # - logits è un tensor di forma (batch_size, 10)
        # - Per ogni immagine, abbiamo 10 numeri (uno per ogni classe: T-shirt, Trouser, ...)
        # - Questi numeri sono detti "logits" o "punteggi grezzi"
        # - Possono essere:
        #   * Positivi o negativi
        #   * Grandi o piccoli
        #   * Non sono ancora probabilità (non sommano a 1)

        return logits
        # return logits: restituisce i logits al chiamante

        # IMPORTANTE: NON applichiamo softmax qui!
        # Perché non usiamo softmax nell'forward?

        # 1. La funzione di loss nn.CrossEntropyLoss in PyTorch:
        #    - Include già internamente la softmax
        #    - Si aspetta logits, non probabilità
        #    - Se applicassimo softmax qui, faremmo softmax due volte

        # 2. Stabilità numerica:
        #    - La softmax può causare problemi con valori estremi
        #    - CrossEntropyLoss usa trucchi numerici che funzionano meglio con logits

        # 3. Flessibilità:
        #    - A volte vuoi i logits per altri scopi (es. calcolare metriche diverse)
        #    - Puoi sempre applicare softmax dopo se necessario:
        #      probabilità = torch.softmax(logits, dim=1)

        # COSA RESTITUISCE forward():
        # Per batch_size=64: tensor di forma (64, 10)
        # Esempio per una singola immagine:
        # [2.1, -1.3, 0.5, 3.2, -0.8, 1.7, -2.1, 0.3, -0.5, 1.2]
        # Il valore più alto (3.2) corrisponde alla classe predetta (es. classe 3 = Dress)

We create an instance of NeuralNetwork, and move it to the device, and print its structure.

In [18]:
model = NeuralNetwork().to(device)
# model = NeuralNetwork().to(device): crea un'istanza del modello e la sposta sul dispositivo scelto

# ANALIZZIAMO IN DUE PARTI:

# 1. NeuralNetwork()
#    - Crea un'istanza della classe NeuralNetwork che abbiamo definito prima
#    - Viene chiamato il costruttore __init__ che:
#      * Inizializza self.flatten = nn.Flatten()
#      * Inizializza self.linear_relu_stack = nn.Sequential(...)
#      * Prepara tutti i layer e i parametri (pesi e bias)
#    - I pesi sono inizializzati automaticamente (default di PyTorch)

# 2. .to(device)
#    - Sposta TUTTO il modello sul dispositivo specificato (es. "cuda", "mps", "cpu")
#    - Sposta:
#      * Tutti i parametri del modello (pesi e bias)
#      * I buffer (es. running_mean in BatchNorm)
#    - Se device = "cuda", tutto va sulla GPU
#    - Se device = "cpu", tutto rimane sulla CPU
#    - Importante: se non fai .to(device), il modello rimane su CPU

print(model)
# print(model): stampa una rappresentazione testuale dell'architettura del modello

# Output tipico:
# NeuralNetwork(
#   (flatten): Flatten(start_dim=1, end_dim=-1)
#   (linear_relu_stack): Sequential(
#     (0): Linear(in_features=784, out_features=512, bias=True)
#     (1): ReLU()
#     (2): Linear(in_features=512, out_features=512, bias=True)
#     (3): ReLU()
#     (4): Linear(in_features=512, out_features=10, bias=True)
#   )
# )

# COSA CI DICE QUESTO OUTPUT:
# - Mostra tutti i layer del modello con i loro nomi
# - Per i layer Linear:
#   * in_features: dimensione input (784 pixel, 512 neuroni, ecc.)
#   * out_features: dimensione output (512, 512, 10)
#   * bias=True: c'è un termine di bias addestrabile
# - La struttura sequenziale è chiara con i numeri (0), (1), (2), ...

# COSA CONTIENE ORA model:
# - model.flatten: il layer per appiattire le immagini
# - model.linear_relu_stack: il contenitore sequenziale
# - model.parameters(): tutti i pesi e bias del modello
#   * Circa 669.194 parametri addestrabili
# - model.to(device): assicura che tutto sia sul dispositivo giusto

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


To use the model, we pass it the input data. This executes the model’s forward, along with some background operations. Do not call model.forward() directly!

Calling the model on the input returns a 2-dimensional tensor with dim=0 corresponding to each output of 10 raw predicted values for each class, and dim=1 corresponding to the individual values of each output. We get the prediction probabilities by passing it through an instance of the nn.Softmax module.

In [19]:
X = torch.rand(1, 28, 28, device=device)
# X = torch.rand(1, 28, 28, device=device): crea un tensor casuale simulando un'immagine
# - torch.rand(): genera valori casuali uniformi tra 0 e 1
# - (1, 28, 28): forma del tensor
#   * 1: una sola immagine (batch_size=1)
#   * 28: altezza in pixel
#   * 28: larghezza in pixel
# - NOTA: manca la dimensione del canale! Dovrebbe essere (1, 1, 28, 28) per Fashion-MNIST
#   * Il modello si aspetta (batch, canali, altezza, larghezza)
#   * Qui abbiamo solo (batch, altezza, larghezza) - potrebbe funzionare lo stesso per broadcasting?
# - device=device: crea il tensor direttamente sul dispositivo scelto (GPU/CPU)

logits = model(X)
# logits = model(X): passaggio in avanti (forward pass)
# - model(X) chiama automaticamente il metodo forward (che dice come procedere nella rete) della nostra NeuralNetwork
# - X passa attraverso:
#   * flatten: da (1, 28, 28) → (1, 784) se X è (1,28,28) senza canale
#   * linear_relu_stack: trasformazioni non lineari
# - Output logits: tensor di forma (1, 10) - punteggi grezzi per 10 classi
# - Una riga (per l'unica immagine) e 10 colonne (una per classe)

pred_probab = nn.Softmax(dim=1)(logits)
# pred_probab = nn.Softmax(dim=1)(logits): converte logits in probabilità
# - nn.Softmax(dim=1): crea un layer Softmax che opera sulla dimensione 1 (le classi)
# - (logits): applica Softmax ai logits
# - Softmax: trasforma i logits in probabilità che sommano a 1
#   * formula: exp(logits_i) / sum_j exp(logits_j)
#   * Garantisce che tutti i valori siano positivi e sommino a 1
# - Output: (1, 10) con valori che rappresentano probabilità per ogni classe
# - Esempio: [0.01, 0.02, 0.05, 0.70, 0.03, 0.05, 0.02, 0.04, 0.03, 0.05]
#   * La classe 3 ha probabilità 70%

y_pred = pred_probab.argmax(1)
# y_pred = pred_probab.argmax(1): trova la classe con probabilità massima
# - .argmax(1): restituisce l'indice del valore massimo lungo la dimensione 1 (le classi)
# - Per l'esempio sopra, il massimo è 0.70 alla posizione 3
# - Output: tensor([3]) - predice che l'immagine è della classe 3 (Dress)
# - Shape: (1,) - un singolo valore per l'unica immagine

print(f"Predicted class: {y_pred}")
# print(f"Predicted class: {y_pred}"): stampa la classe predetta
# - Output esempio: "Predicted class: tensor([3], device='cuda:0')"
# - Mostra l'indice della classe predetta e su quale dispositivo si trova

Predicted class: tensor([3])
